<div style="background-color: #ffffff; color: #000000; padding: 10px;">
<div style="display: flex; justify-content: space-between; align-items: center; background-color: #ffffff; color: #000000; padding: 10px;">
    <img src="../00_aisc/img/logo_aisc_150dpi.png" height="80" style="margin-right: auto;" alt="Logo of the AI Service Center Berlin-Brandenburg.">
    <img src="../00_aisc/img/logo_bmftr_de.png" height="150" style="margin-left: auto;" alt="Logo of the German Federal Ministry of Research, Technology and Space: Gefördert vom Bundesministerium für Forschung, Technologie und Raumfahrt.">
</div>
<h1> RAG III
</div>

# 03 — Retrieval Evaluation

Dieses Notebook evaluiert die **Retrieval-Qualität** unseres RAG-Systems in Isolation —
bevor eine Antwort generiert wird.

1. Ground-Truth-Daten laden (CSV mit kuratierten Fragen & Referenzantworten)
2. Fragen einbetten und relevante Chunks aus Qdrant abrufen
3. **Context Precision** messen — Stehen relevante Chunks oben im Ranking?
4. **Context Recall** messen — Deckt das Retrieval alle Informationen der Referenzantwort ab?
5. Ergebnisse inspizieren
6. Experiment: Systematischer Vergleich verschiedener `TOP_K`-Werte

> **Keine Antwortgenerierung** in diesem Notebook — wir bewerten nur das Retrieval.

**Voraussetzung:** `w3_02_ingestion.ipynb` muss erfolgreich durchgelaufen sein (Qdrant-Collection befüllt).

In [ ]:
# Gemeinsame Konfiguration laden (Pfade, Modellnamen, Qdrant-Einstellungen)
from config import *

# --- Imports ---
import json
import os
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display
from typing import List
from openai import AsyncOpenAI
from litellm import embedding
from qdrant_client import QdrantClient
from ragas.llms import llm_factory
from ragas.metrics.collections import ContextPrecision, ContextRecall
from ragas.metrics.collections.context_precision.metric import ContextPrecisionInput, ContextPrecisionOutput
from ragas.metrics.collections.context_recall.util import ContextRecallInput, ContextRecallOutput

## 1) Ground Truth laden

Wir verwenden einen von Fachexperten kuratierten Datensatz mit rund 40 Fragen zum IT-Grundschutz-Kompendium.
Jede Frage hat:
- **Frage** → `user_input` — Was wird an das System gestellt?
- **Antwort** → `reference` — Die korrekte Referenzantwort
- **Fundstellen** → Die relevanten Passagen im Originaldokument (für manuelle Inspektion)

In [ ]:
# CSV einlesen (Datensatz und Trennzeichen aus config.py)
df = pd.read_csv(CSV_PATH, sep=CSV_SEP)

questions = df['Frage'].tolist()       # Alle Fragen als Liste von Strings
references = df['Antwort'].tolist()    # Zugehörige Referenzantworten als Liste von Strings

print(f'{len(df)} Fragen geladen (Datensatz: {DATASET}).')
print(f'Spalten: {list(df.columns)}\n')
df.head(3)

## 2) Retrieval: Kontexte aus Qdrant abrufen

Für jede Frage wird:
1. Die Frage mit dem Embedding-Modell in einen Vektor umgewandelt
2. Per Cosine Similarity die `TOP_K` ähnlichsten Chunks aus Qdrant abgerufen

Die Qdrant-Collection wurde in `w3_02_ingestion.ipynb` befüllt.

In [ ]:
def _truncate(text: str, max_chars: int = 500) -> str:
    """Truncate text to max_chars, breaking at the last space if possible.

    Args:
        text: Input text to truncate.
        max_chars: Maximum number of characters.

    Returns:
        Truncated text.
    """
    if len(text) <= max_chars:
        return text
    truncated = text[:max_chars]
    last_space = truncated.rfind(" ")
    if last_space > max_chars // 2:
        truncated = truncated[:last_space]
    return truncated


def embed_texts_litellm(texts: List[str], model: str = EMBED_MODEL_NAME, batch_size: int = EMBED_BATCH_SIZE, max_chars: int = EMBED_MAX_CHARS) -> List[List[float]]:
    """Embed a list of texts using the HPI API via LiteLLM.

    Args:
        texts: List of text strings to embed.
        model: Model identifier in LiteLLM format.
        batch_size: Number of texts to embed per API call.
        max_chars: Truncate texts to this many characters before embedding (None = no limit).

    Returns:
        List of embedding vectors (each a list of floats).
    """
    if max_chars:
        texts = [_truncate(t, max_chars) for t in texts]
    vectors = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i + batch_size]                # Aktuellen Batch aus der Textliste schneiden
        resp = embedding(model=model, input=batch, api_base=API_BASE_URL, encoding_format='float')  # API-Aufruf: Texte → Vektoren
        for item in resp.data:
            # LiteLLM gibt Embeddings je nach Version als Objekt oder Dict zurück
            vec = item.embedding if hasattr(item, 'embedding') else item.get('embedding')
            if vec is None:
                raise RuntimeError('Embedding response ohne Vektor erhalten.')
            vectors.append(vec)
        print(f'  Embedded {len(vectors)}/{len(texts)}', end='\r')  # Fortschrittsanzeige
    print()
    return vectors

# Verbindung zu Qdrant herstellen (Collection wurde in Notebook 02 befüllt)
qdrant = QdrantClient(host=QDRANT_HOST, port=QDRANT_PORT)
info = qdrant.get_collection(COLLECTION_NAME)
print(f'Qdrant-Collection "{COLLECTION_NAME}": {info.points_count} Vektoren\n')

# Fragen in Vektoren umwandeln (gleicher Embedding-Raum wie die Chunks)
print(f'Embedding {len(questions)} Fragen...')
question_vectors = embed_texts_litellm(questions)

# Für jede Frage die TOP_K ähnlichsten Chunks aus Qdrant per Cosine-Similarity abrufen
print(f'Retrieving top-{TOP_K} Kontexte pro Frage...')
retrieved_contexts = []
for q_vec in question_vectors:
    response = qdrant.query_points(
        collection_name=COLLECTION_NAME,
        query=q_vec,                    # Frage-Vektor als Suchanfrage
        limit=TOP_K,                    # Anzahl der Treffer
        with_payload=True,              # Chunk-Text im Ergebnis mitliefern
    )
    # Nur den Textinhalt der Treffer extrahieren (aus dem Payload)
    contexts = [hit.payload['text'] for hit in response.points]
    retrieved_contexts.append(contexts)

print(f'Fertig. {len(retrieved_contexts)} Fragen mit je {TOP_K} Kontexten.')

In [ ]:
# Sanity Check: erste Frage und ihre Top-Treffer
print("="*120+"")
print(f'Frage: {questions[0]}')
print(f'Referenzantwort: {references[0][:120]}...')
print("="*120+"\n")
for i, ctx in enumerate(retrieved_contexts[0]):
    print(f'--------------- Top-{i+1} Chunk: ---------------')
    print(f'{ctx[:120]}{"..." if len(ctx) > 120 else ""}')
    print()
    if i == 2: # only print out the first three 
        break

## Wie bewertet RAGAS? — LLM-as-Judge

Bevor wir die einzelnen Metriken betrachten, ist es wichtig zu verstehen, **wie** RAGAS
seine Bewertungen durchführt. RAGAS verwendet ein **Evaluator-LLM als Richter**.

## 3) Context Precision — Sind die abgerufenen Chunks relevant?

Misst, ob relevante Chunks weiter **oben** im Ranking stehen als irrelevante.
Für jeden abgerufenen Chunk fragt das Evaluator-LLM: *„Ist dieser Chunk nützlich,
um die Frage zu beantworten?"* (Verdict: 1=Ja, 0=Nein)

Daraus wird **Mean Average Precision** berechnet (relevante Chunks weiter oben = höherer Score):

| Position | Relevant? | Precision@K |
|----------|-----------|-------------|
| 1        | Ja        | 1/1 = 1.0   |
| 2        | Nein      | —           |
| 3        | Ja        | 2/3 = 0.67  |

→ Context Precision = (1.0 + 0.67) / 2 = **0.84**

**Benötigte Felder:** `user_input`, `retrieved_contexts`, `reference`

In [ ]:
# Evaluator-LLM einrichten
evaluator_client = AsyncOpenAI(
    api_key=os.getenv('OPENAI_API_KEY'),
    base_url=API_BASE_URL,
)
evaluator_llm = llm_factory(
    EVALUATOR_MODEL_NAME,
    client=evaluator_client,
    max_tokens=8192,
)

# Metriken initialisieren
context_precision = ContextPrecision(llm=evaluator_llm)
context_recall = ContextRecall(llm=evaluator_llm)

# ---------------------------------------------------------------------------
# Demo: Context Precision für eine einzelne Frage — Blick unter die Haube
# ---------------------------------------------------------------------------
# Für jeden abgerufenen Chunk fragt das Evaluator-LLM:
# "Ist dieser Chunk nützlich, um die Frage zu beantworten?" (Verdict: 1=Ja, 0=Nein)
# ---------------------------------------------------------------------------

demo_idx = 3
print(f'Frage:    {questions[demo_idx]}')
print(f'Referenz: {references[demo_idx][:120]}...\n')

# Jeden Chunk einzeln bewerten
verdicts = []
for i, ctx in enumerate(retrieved_contexts[demo_idx]):
    input_data = ContextPrecisionInput(
        question=questions[demo_idx],
        context=ctx,
        answer=references[demo_idx],
    )
    prompt_str = context_precision.prompt.to_string(input_data)
    result = await evaluator_llm.agenerate(prompt_str, ContextPrecisionOutput)
    verdicts.append(result.verdict)
    icon = '✓ relevant' if result.verdict else '✗ irrelevant'
    print(100*"-")
    print(f'  Chunk {i+1}: [{icon}]')
    print(f'           Grund: {result.reason}')

# --- Average Precision Schritt für Schritt berechnen ---
# Formel: Für jede Position k mit Verdict=1 berechne Precision@k,
# dann Durchschnitt über alle relevanten Positionen.
print('=' * 100)
print(f'Verdicts: {verdicts}\n')


# calculate context precision manually
relevant_precisions = []
for k in range(len(verdicts)):
    if verdicts[k] == 1:
        precision_at_k = sum(verdicts[:k+1]) / (k + 1)
        relevant_precisions.append(precision_at_k)
        print(f'  Position {k+1}: relevant → Precision@{k+1} = {sum(verdicts[:k+1])}/{k+1} = {precision_at_k:.3f}')
    else:
        print(f'  Position {k+1}: irrelevant → wird übersprungen')

if relevant_precisions:
    avg_precision = sum(relevant_precisions) / len(relevant_precisions)
    terms = ' + '.join(f'{p:.2f}' for p in relevant_precisions)
    print(f'\n  Context Precision = ({terms}) / {len(relevant_precisions)} = {avg_precision:.2f}')
else:
    print(f'\n  Kein relevanter Chunk gefunden → Context Precision = 0.0')

# Zur Kontrolle: RAGAS-interne Berechnung mit der gleichen Funktion
score_automated = context_precision._calculate_average_precision(verdicts)
print(f'  Kontrolle (RAGAS intern): {score_automated:.2f}')

Bevor wir zur nächsten Metrik übergehen können, müssen wir zunächst erklären was "claims" in RAGAS sind.

### Claims: Die Grundeinheit der Bewertung

Fast alle RAGAS-Metriken basieren auf dem Konzept von **Claims** (Aussagen).
Das Evaluator-LLM zerlegt einen Text in atomare, eigenständige Aussagen. Jede Aussage sagt möglichst 
einen isolierten Fakt aus, sodass sie einzeln verständlich ist.

**Beispiel:** Der Evaluator erhält folgenden Text:

> *„Der ISB erstellt die Sicherheitsleitlinie und koordiniert Maßnahmen mit den Fachabteilungen."*

Daraus erzeugt er zwei Claims:
1. *„Der ISB erstellt die Sicherheitsleitlinie."*
2. *„Der ISB koordiniert Maßnahmen mit den Fachabteilungen."*

Anschließend wird für jede Aussage geprüft, ob sie durch einen anderen Text belegt ist
(z.B. durch die abgerufenen Chunks oder die Referenzantwort — je nach Metrik).

### Warum das wichtig ist

- Es gibt **keinen Algorithmus** hinter der Zerlegung — das Evaluator-LLM entscheidet,
  was eine „atomare Aussage" ist und ob sie belegt ist.
- **Verschiedene Evaluator-Modelle liefern verschiedene Ergebnisse:** Ein stärkeres Modell
  erkennt feinere Unterschiede und bewertet strenger. Ein schwächeres Modell ist unkritischer und produziert daher eher höhere Scores.
- Die **Anzahl der Claims** beeinflusst den Score direkt: Wenn das LLM einen Satz in 3 statt 5
  Claims zerlegt, hat eine einzelne unbelegte Aussage mehr Gewicht (1/3 vs. 1/5).
- RAGAS-Scores sind daher **keine objektiven Messwerte**, sondern **LLM-basierte Urteile** —
  vergleichbar nur, wenn dasselbe Evaluator-Modell verwendet wurde. Und selbst dann unterscheiden sich Scores bei erneuter Berechnung da die Identifikation der claims nicht deterministisch ist.

## 4) Context Recall — Deckt das Retrieval alle Informationen ab?

Misst, ob die abgerufenen Chunks genug Information enthalten, um die Referenzantwort zu belegen.

**So funktioniert die Metrik:**
1. Das Evaluator-LLM zerlegt die **Referenzantwort** in atomare Claims
2. Für jeden Claim prüft es: *„Kann diese Aussage den abgerufenen Chunks zugeordnet werden?"*
3. Score = Anzahl belegter Claims / Gesamtanzahl Claims

$$ \text{Context Recall} = \frac{\text{Anzahl Referenz-Claims, die in den Chunks belegt sind}}{\text{Gesamtanzahl Referenz-Claims}} $$

**Benötigte Felder:** `user_input`, `retrieved_contexts`, `reference`

### Zusammenspiel von Context Precision und Context Recall

| | Precision hoch | Precision niedrig |
|---|---|---|
| **Recall hoch** | Ideal: relevante Chunks, gut gerankt | Viele Chunks nötig, aber alles abgedeckt |
| **Recall niedrig** | Wenige, gute Chunks — aber Info fehlt | Schlechtes Retrieval insgesamt |

In [ ]:
# ---------------------------------------------------------------------------
# Demo: Context Recall für eine einzelne Frage — Blick unter die Haube
# ---------------------------------------------------------------------------
# Das Evaluator-LLM zerlegt die Referenzantwort in Claims und prüft für jeden:
# "Kann diese Aussage den abgerufenen Chunks zugeordnet werden?" (1=Ja, 0=Nein)
#
# WICHTIG: Separater LLM-Aufruf — Claims können bei jedem Durchlauf variieren.
# ---------------------------------------------------------------------------

from ragas.metrics.collections.context_recall.util import ContextRecallInput, ContextRecallOutput

demo_idx = 4 
context_str = '\n'.join(retrieved_contexts[demo_idx])

print(f'Frage:    {questions[demo_idx]}')
print(f'Referenz: {references[demo_idx][:120]}...\n')

# Context Recall intern aufrufen: Referenz in Claims zerlegen + gegen Chunks prüfen
input_data = ContextRecallInput(
    question=questions[demo_idx],
    context=context_str,
    answer=references[demo_idx],
)
prompt_str = context_recall.prompt.to_string(input_data)
result = await evaluator_llm.agenerate(prompt_str, ContextRecallOutput)

# Ergebnisse anzeigen
n_attributed = sum(1 for c in result.classifications if c.attributed)
print(f'Die Referenzantwort wurde in {len(result.classifications)} Claims zerlegt:\n')

for i, c in enumerate(result.classifications):
    icon = '✓ belegt' if c.attributed else '✗ NICHT belegt'
    print(f'  {i+1}. [{icon}] {c.statement}')
    print(f'     Grund: {c.reason}')
    print()

print(f'Context Recall = {n_attributed}/{len(result.classifications)} = {n_attributed/len(result.classifications):.3f}')

Was passiert wenn wir die obrige Codezelle erneut ausführen? Bleibt die Anzahl der claims gleich? 

Nun können wir Context Precision und Recall für alle Fragen im Datensatz berechnen.

In [ ]:
# ---------------------------------------------------------------------------
# Batch-Evaluation: Context Precision + Context Recall für alle Fragen
# ---------------------------------------------------------------------------
# Evaluator-LLM und Metriken sind bereits oben initialisiert.
# ---------------------------------------------------------------------------

LOAD_FROM_CACHE = True
CACHE_PATH = CACHE_DIR / f'retrieval_eval_top{TOP_K}.json'

if LOAD_FROM_CACHE and CACHE_PATH.exists():
    with open(CACHE_PATH, 'r') as f:
        cached = json.load(f)
    cp_scores = cached['context_precision']
    cr_scores = cached['context_recall']
    print(f'Ergebnisse aus Cache geladen: {CACHE_PATH.name}')
else:
    inputs = [
        {
            'user_input': questions[i],
            'reference': references[i],
            'retrieved_contexts': retrieved_contexts[i],
        }
        for i in range(len(questions))
    ]

    print(f'Evaluiere Context Precision für {len(questions)} Fragen (parallel)...')
    cp_results = await context_precision.abatch_score(inputs)
    cp_scores = [float(r.value) for r in cp_results]

    print(f'Evaluiere Context Recall für {len(questions)} Fragen (parallel)...')
    cr_results = await context_recall.abatch_score(inputs)
    cr_scores = [float(r.value) for r in cr_results]

    with open(CACHE_PATH, 'w') as f:
        json.dump({'context_precision': cp_scores, 'context_recall': cr_scores}, f)
    print(f'Ergebnisse gespeichert: {CACHE_PATH.name}')

# DataFrame aufbauen
result_df = pd.DataFrame({
    'user_input': questions,
    'reference': references,
    'context_precision': cp_scores,
    'context_recall': cr_scores,
})
result_df['retrieved_contexts'] = retrieved_contexts

print(f'\nContext Precision (Durchschnitt): {result_df["context_precision"].mean():.4f}')
print(f'Context Recall (Durchschnitt):    {result_df["context_recall"].mean():.4f}')

## 5) Ergebnisse inspizieren

In [ ]:
# Histogramme für beide Metriken nebeneinander
metric_cols = ['context_precision', 'context_recall']
fig, axes = plt.subplots(2, 1, figsize=(9, 5))

for ax, col in zip(axes, metric_cols):
    ax.hist(result_df[col], bins=40, edgecolor='black')
    ax.set_xlabel('Score')
    ax.set_ylabel('Count')
    mean_score = result_df[col].mean()
    ax.set_title(f'{col}')
    ax.axvline(mean_score, color='red', linestyle='--', label=f'Mean: {mean_score:.3f}')
    ax.legend()

fig.suptitle(
    f'{EMBED_SHORT} | {CHUNKING_MODE} | {DATASET} | top_k={TOP_K} | eval={EVALUATOR_SHORT}',
    fontsize=13, y=1.03,
)
plt.tight_layout()

# Figur speichern
fig_dir = FIGURES_DIR / '03_retrieval_evaluation'
fig_dir.mkdir(parents=True, exist_ok=True)
fig_path = fig_dir / f'retrieval_hist__{EMBED_SHORT}__{CHUNKING_MODE}__{DATASET}__top{TOP_K}__{EVALUATOR_SHORT}.png'
fig.savefig(fig_path, dpi=150, bbox_inches='tight')
print(f'Saved: {fig_path}')

plt.show()

## 6) Experiment: TOP_K systematisch variieren

Was passiert mit Context Precision und Context Recall, wenn wir `TOP_K` verändern?

- **Precision** sinkt tendenziell mit höherem TOP_K (mehr irrelevante Chunks im Ranking)
- **Recall** steigt tendenziell mit höherem TOP_K (mehr Kontext = mehr Informationsabdeckung)

Die optimale Wahl von TOP_K balanciert beide Metriken.

> **Laufzeit:** ~2 Minuten pro TOP_K-Wert (4 Werte × 2 Metriken). Mit `LOAD_FROM_CACHE = True` sofort.

In [ ]:
# --- TOP_K-Werte für das Experiment ---
TOP_K_VALUES = [1, 3, 5, 10]
EXPERIMENT_CACHE = CACHE_DIR / 'retrieval_experiment_topk.json'  # CACHE_DIR enthält bereits Datensatz + Chunking + Embedding + Evaluator

if LOAD_FROM_CACHE and EXPERIMENT_CACHE.exists():
    # Gespeicherte Experiment-Ergebnisse laden
    with open(EXPERIMENT_CACHE, 'r') as f:
        experiment_results = json.load(f)
    print(f'Experiment-Ergebnisse aus Cache geladen: {EXPERIMENT_CACHE.name}')
else: 
    experiment_results = {}

    for top_k in TOP_K_VALUES:
        print(f'=== TOP_K={top_k} ===')

        # Retrieval mit aktuellem TOP_K
        print(f'  Retrieval...')
        exp_contexts = []
        for q_vec in question_vectors:
            response = qdrant.query_points(
                collection_name=COLLECTION_NAME,
                query=q_vec,
                limit=top_k,
                with_payload=True,
            )
            exp_contexts.append([hit.payload['text'] for hit in response.points])

        # Eingaben für Batch-Evaluation aufbauen
        inputs = [
            {
                'user_input': questions[i],
                'reference': references[i],
                'retrieved_contexts': exp_contexts[i],
            }
            for i in range(len(questions))
        ]

        # Context Precision bewerten
        print(f'  Context Precision...')
        cp_results = await context_precision.abatch_score(inputs)
        cp_scores = [float(r.value) for r in cp_results]

        # Context Recall bewerten
        print(f'  Context Recall...')
        cr_results = await context_recall.abatch_score(inputs)
        cr_scores = [float(r.value) for r in cr_results]

        experiment_results[str(top_k)] = {
            'context_precision': cp_scores,
            'context_recall': cr_scores,
        }

        avg_cp = sum(cp_scores) / len(cp_scores)
        avg_cr = sum(cr_scores) / len(cr_scores)
        print(f'  → Precision: {avg_cp:.4f}, Recall: {avg_cr:.4f}\n')

    # Ergebnisse speichern
    with open(EXPERIMENT_CACHE, 'w') as f:
        json.dump(experiment_results, f)
    print(f'Experiment-Ergebnisse gespeichert: {EXPERIMENT_CACHE.name}')

In [ ]:
# Liniendiagramm: Context Precision und Context Recall vs TOP_K
fig, ax = plt.subplots(figsize=(8, 5))

top_k_vals = [int(k) for k in experiment_results.keys()]  # String-Keys → int
avg_precision = [sum(v['context_precision']) / len(v['context_precision']) for v in experiment_results.values()]
avg_recall = [sum(v['context_recall']) / len(v['context_recall']) for v in experiment_results.values()]

ax.plot(top_k_vals, avg_precision, 'o-', label='Context Precision', linewidth=2, markersize=8)
ax.plot(top_k_vals, avg_recall, 's-', label='Context Recall', linewidth=2, markersize=8)

ax.set_xlabel('TOP_K')
ax.set_ylabel('Mean Score')
ax.set_title(f'Context Precision vs. Recall \n {EMBED_SHORT} | {CHUNKING_MODE} | {DATASET} | eval={EVALUATOR_SHORT}')
ax.set_xticks(top_k_vals)
ax.set_ylim(0,1)
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()

# Figur speichern
fig_dir = FIGURES_DIR / '03_retrieval_evaluation'
fig_dir.mkdir(parents=True, exist_ok=True)
fig_path = fig_dir / f'topk_precision_recall__{EMBED_SHORT}__{CHUNKING_MODE}__{DATASET}__{EVALUATOR_SHORT}.png'
fig.savefig(fig_path, dpi=150, bbox_inches='tight')
print(f'Saved: {fig_path}')

plt.show()

In [ ]:
# Histogramm-Grid: eine Zeile pro Metrik, eine Spalte pro TOP_K
metrics = ['context_precision', 'context_recall']
n_topk = len(TOP_K_VALUES)

fig, axes = plt.subplots(2, n_topk, figsize=(3 * n_topk, 5))

for row, metric in enumerate(metrics):
    for col, top_k in enumerate(TOP_K_VALUES):
        ax = axes[row, col]
        scores = experiment_results[str(top_k)][metric]
        ax.hist(scores, bins=20, edgecolor='black', alpha=0.7)
        avg = sum(scores) / len(scores)
        ax.axvline(avg, color='red', linestyle='--', linewidth=1, label=f'Mean: {avg:.2f}')
        ax.set_xlim(0, 1.05)
        ax.set_title(f'TOP_K={top_k}', fontsize=10)
        ax.legend()
        if col == 0:
            ax.set_ylabel(metric.replace('_', '\n'), fontsize=9)
        if row == 1:
            ax.set_xlabel('Score', fontsize=9)

fig.suptitle(
    f'Score Distributions \n {EMBED_SHORT} | {CHUNKING_MODE} | {DATASET} | eval={EVALUATOR_SHORT}'
)
plt.tight_layout()

# Figur speichern
fig_dir = FIGURES_DIR / '03_retrieval_evaluation'
fig_dir.mkdir(parents=True, exist_ok=True)
fig_path = fig_dir / f'topk_hist_grid__{EMBED_SHORT}__{CHUNKING_MODE}__{DATASET}__{EVALUATOR_SHORT}.png'
fig.savefig(fig_path, dpi=150, bbox_inches='tight')
print(f'Saved: {fig_path}')

plt.show()

## Diskussion

- Welcher TOP_K-Wert balanciert Precision und Recall am besten?
- Welche Aussagekraft hat Context Precision bei TOP_K = 1?
- Welche Aussagekraft hat Context Recall bei einem hohen TOP_K Wert?
- Wie verändert sich die Score-Verteilung (nicht nur der Durchschnitt) mit steigendem TOP_K?

> **Tipp:** In `config.py` können die Parameter umgeschaltet werden: Embedding Modell, Chunking Methode, Dataset, Evaluator LLM.

- was hat den größten Effekt auf die Scores? Embedding Modell? Chunking Methode? Dataset? Evaluator LLM?
- Was könnte man am Retrieval verbessern?

**Weiter geht es mit:**
- `w3_04_generation_evaluation.ipynb` — Generations-Metriken (Faithfulness und Noise Sensitivity)